# Yahoo Finance

yfinance is a popular Python library that enables users to access historical market data from Yahoo Finance with ease. It allows for the retrieval of stock prices, financial statements, dividends, splits, and more using a simple, intuitive API. Designed for both casual users and professional analysts, yfinance supports downloading data as Pandas DataFrames, making it ideal for financial analysis, algorithmic trading, and machine learning applications.

## Install the library

In [ ]:
!pip install yfinance

## Load the data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import seaborn as sb
import statsmodels.api as sm

In [ ]:
tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'BRK-B', 'JPM', 'V',
    'JNJ', 'WMT', 'PG', 'UNH', 'HD', 'MA', 'DIS', 'PEP', 'KO', 'XOM',
    'PFE', 'BAC', 'CSCO', 'CVX', 'ABBV', 'CRM', 'INTC', 'ABT', 'T', 'VZ',
    'ORCL', 'ADBE', 'NKE', 'TMO', 'LLY', 'QCOM', 'TXN', 'COST', 'MCD', 'UPS',
    'LOW', 'MDT', 'AVGO', 'BA', 'GS', 'PYPL', 'HON', 'DHR', 'UNP', 'AMAT'
]

data = []

for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    bs = t.balance_sheet  # quarterly by default, latest column is first

    # Try to get total assets from most recent balance sheet
    try:
        total_assets = bs.loc['Total Assets'][0]
    except Exception:
        total_assets = None

    try:
        rd_expense = t.financials.loc['Research Development'][0]
    except:
        rd_expense = None

    try:
        total_revenue = t.financials.loc['Total Revenue'][0]
    except:
        total_revenue = None

    data.append({
        'ticker': ticker,
        'sector': info.get('sector'),
        'industry': info.get('industry'),
        'profitMargins': info.get('profitMargins'),
        'returnOnAssets': info.get('returnOnAssets'),
        'grossMargins': info.get('grossMargins'),
        'operatingMargins': info.get('operatingMargins'),
        'totalAssets': total_assets,
        'totalRevenue': total_revenue,
        'summary': info.get('longBusinessSummary', '')
    })

df = pd.DataFrame(data)
df = df.dropna(subset=['returnOnAssets', 'grossMargins', 'totalAssets'])
df['logAssets'] = np.log(df['totalAssets'])

# Optional categorization
df['businessModel'] = 'Other'
df.loc[df['summary'].str.contains('software|cloud', case=False, na=False), 'businessModel'] = 'SaaS'
df.loc[df['summary'].str.contains('retail|store', case=False, na=False), 'businessModel'] = 'Retail'

/tmp/ipython-input-31-2737639612.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_assets = bs.loc['Total Assets'][0]
/tmp/ipython-input-31-2737639612.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_revenue = t.financials.loc['Total Revenue'][0]
/tmp/ipython-input-31-2737639612.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_assets = bs.loc['Total Assets'][0]
/tmp/ipython-input-31-2737639612.py:29: Fut

In [ ]:
df.head()

,ticker,sector,industry,profitMargins,returnOnAssets,grossMargins,operatingMargins,totalAssets,totalRevenue,summary,logAssets,businessModel
0,AAPL,Technology,Consumer Electronics,0.24301,0.23810,0.46632,0.31029,3.649800e+11,3.910350e+11,"Apple Inc. designs, manufactures, and markets ...",26.623108,Retail
1,MSFT,Technology,Software - Infrastructure,0.35789,0.14582,0.69075,0.45671,5.121630e+11,2.451220e+11,Microsoft Corporation develops and supports so...,26.961909,Retail
2,GOOGL,Communication Services,Internet Content & Information,0.30857,0.16897,0.58591,0.33918,4.502560e+11,3.500180e+11,Alphabet Inc. offers various products and plat...,26.833082,SaaS
3,AMZN,Consumer Cyclical,Internet Retail,0.10140,0.07632,0.49158,0.11823,6.248940e+11,6.379590e+11,"Amazon.com, Inc. engages in the retail sale of...",27.160848,Retail
4,META,Communication Services,Internet Content & Information,0.39114,0.17880,0.81767,0.41487,2.760540e+11,1.645010e+11,"Meta Platforms, Inc. engages in the developmen...",26.343862,SaaS


## Linear Regression: Predicting ROA from Gross Margins and Log Assets

In [ ]:
tmp = df[['returnOnAssets', 'grossMargins', 'logAssets']].dropna()
X = tmp[['grossMargins', 'logAssets']]

# use X with sm.OLS
model = sm.OLS(tmp['returnOnAssets'], sm.add_constant(X))
bs = model.fit().params
bs

,0
const,0.373696
grossMargins,0.131057
logAssets,-0.013341


The model shows a significant positive relationship between grossMargins and return on assets, while logAssets has a small negative association.

## Correlation

In [ ]:
tmp = df[['returnOnAssets', 'grossMargins', 'logAssets', 'profitMargins']].dropna()
X = tmp[['grossMargins', 'logAssets', 'profitMargins']]

# use X with sm.OLS
model = sm.OLS(tmp['returnOnAssets'], sm.add_constant(X))
bs = model.fit().params
bs

,0
const,0.662715
grossMargins,-0.005763
logAssets,-0.024353
profitMargins,0.359536


In [ ]:
df['profitMargins'].corr(df['grossMargins'])

np.float64(0.5406819526029887)

Including profitMargins improves the model’s prediction of return on assets, showing it as a stronger positive predictor than grossMargins, while logAssets remains negatively associated.

## Two Sample t-test

In [ ]:
# Two sample t-test: Difference of means Hypothesis Test for if Saas makes more revenue than Retail
saas = df[df['businessModel'] == 'SaaS']['totalRevenue'].dropna()
retail = df[df['businessModel'] == 'Retail']['totalRevenue'].dropna()
sem = np.sqrt(saas.var()/len(saas) + retail.var()/len(retail))
z = (saas.mean() - retail.mean()) / sem
z

np.float64(-2.407929989856933)